In [4]:
!pip install -q transformers datasets accelerate torch scikit-learn

In [5]:
print("Upgrading 'datasets' and 'huggingface_hub' libraries...")
!pip install -q --upgrade datasets huggingface_hub
print("Libraries upgraded successfully.")
print("Please restart the runtime for changes to take effect.")

Upgrading 'datasets' and 'huggingface_hub' libraries...
Libraries upgraded successfully.
Please restart the runtime for changes to take effect.


In [8]:
import pandas as pd

# Create a dummy CSV file with 'text' and 'label' columns
dummy_data = {
    'text': [
        'This movie was fantastic, I loved every moment!',
        'Absolutely terrible, a waste of time and money.',
        'It was okay, nothing special but not bad either.',
        'Highly recommend this film, great acting and story.',
        'Worst movie of the year, avoid at all costs.',
        'A genuinely heartwarming story that brought tears to my eyes.',
        'Confusing and poorly executed. Did not enjoy.',
        'A true cinematic masterpiece, a must-see for everyone.',
        'Mediocre at best, very predictable plot.',
        'Such a gripping thriller, kept me on the edge of my seat.'
    ],
    'label': [1, 0, 1, 1, 0, 1, 0, 1, 0, 1]
}
dummy_df = pd.DataFrame(dummy_data)
dummy_df.to_csv('local_data.csv', index=False)

print("Created 'local_data.csv' successfully:")
display(dummy_df.head())


Created 'local_data.csv' successfully:


,text,label
0,"This movie was fantastic, I loved every moment!",1
1,"Absolutely terrible, a waste of time and money.",0
2,"It was okay, nothing special but not bad either.",1
3,"Highly recommend this film, great acting and s...",1
4,"Worst movie of the year, avoid at all costs.",0


In [9]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score
import torch

# ============================================================
# 1. LOAD LOCAL CSV DATASET
# ============================================================

# Loading a local CSV file to bypass potential Hugging Face Hub URI issues.
# The 'csv' builder is used, and 'data_files' points to our local CSV.
dataset = load_dataset('csv', data_files='local_data.csv')

# The local CSV will be loaded as a single 'train' split by default.
# We need to manually split it into train and test sets.
# Note: This will create a 'train' key in the `dataset` object.
train_test_split = dataset['train'].train_test_split(test_size=0.2, seed=42)

# Use the same reduced dataset size as the lab experiment
train_data = train_test_split["train"].shuffle(seed=42).select(range(min(2000, len(train_test_split['train']))))
test_data = train_test_split["test"].shuffle(seed=42).select(range(min(500, len(train_test_split['test']))))

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))
print("First training example:", train_data[0])
print("First testing example:", test_data[0])


# ============================================================
# 2. LOAD DISTILBERT TOKENIZER
# ============================================================

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    # The local CSV has a 'text' column.
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_data.map(tokenize, batched=True)
test_ds = test_data.map(tokenize, batched=True)

# Remove original 'text' column
train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

# Tell Hugging Face that labels are required
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

train_ds.set_format("torch")
test_ds.set_format("torch")


# ============================================================
# 3. LOAD PRE-TRAINED DISTILBERT
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


# ============================================================
# 4. TRAINING CONFIGURATION
# ============================================================

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none",
    save_strategy="no"
)


# ============================================================
# 5. EVALUATION METRIC
# ============================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy
    }


# ============================================================
# 6. CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)


# ============================================================
# 7. FINE-TUNE MODEL
# ============================================================

print("\n========== TRAINING ==========")

trainer.train()


# ============================================================
# 8. EVALUATE MODEL
# ============================================================

print("\n========== EVALUATION ==========")

metrics = trainer.evaluate()

print("Accuracy:", round(metrics["eval_accuracy"], 4))
print("Evaluation Loss:", round(metrics["eval_loss"], 4))


# ============================================================
# 9. SAVE FINE-TUNED MODEL
# ============================================================

model.save_pretrained("fine_tuned_distilbert_imdb")
tokenizer.save_pretrained("fine_tuned_distilbert_imdb")

print("\nFine-tuned model saved successfully.")


# ============================================================
# RESULT
# ============================================================

print("\n========== RESULT ==========")
print("The pre-trained DistilBERT model was successfully fine-tuned")
print("on the IMDB sentiment classification dataset.")
print("The model was evaluated and saved successfully.")

Generating train split: 0 examples [00:00, ? examples/s]

Training samples: 8
Testing samples: 2
First training example: {'text': 'It was okay, nothing special but not bad either.', 'label': 1}
First testing example: {'text': 'Confusing and poorly executed. Did not enjoy.', 'label': 0}


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.703503,2,0.500000


Accuracy: 0.5
Evaluation Loss: 0.7035


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved successfully.

========== RESULT ==========
The pre-trained DistilBERT model was successfully fine-tuned
on the IMDB sentiment classification dataset.
The model was evaluated and saved successfully.
